In [0]:
%sql
use catalog datastore

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold

In [0]:
# Read the silver table and alias every column to UPPERCASE
from pyspark.sql.functions import monotonically_increasing_id

df = spark.table("silver.customers")
df_upper = df.select([c.upper() for c in df.columns])

# Add a monotonically increasing ID column
df_dim = df_upper.withColumn("LHID", monotonically_increasing_id())

# Write to the gold layer dimension table (replaces it if it already exists)
df_dim.write.mode("overwrite").saveAsTable("gold.dim_customers")

# Verify
# display(df_upper)

In [0]:

# Read the silver table, uppercase string column values, alias every column name to UPPERCASE,
# and add a monotonically increasing ID column
from pyspark.sql.functions import upper, col, monotonically_increasing_id

df = spark.table("silver.products")

df_upper = df.select([
    upper(c).alias(c.upper()) if df.schema[c].dataType.simpleString() == "string" else col(c).alias(c.upper())
    for c in df.columns
])

# Add a monotonically increasing ID column
df_dim = df_upper.withColumn("LHID", monotonically_increasing_id())

# Write to the gold layer dimension table (replaces it if it already exists)
df_dim.write.mode("overwrite").saveAsTable("gold.dim_products")

# Verify
# display(df_dim)


In [0]:

# Read the silver table, uppercase string column values, alias every column name to UPPERCASE,
# and add a monotonically increasing ID column
from pyspark.sql.functions import upper, col, monotonically_increasing_id

df = spark.table("silver.branches")

df_upper = df.select([
    upper(c).alias(c.upper()) if df.schema[c].dataType.simpleString() == "string" else col(c).alias(c.upper())
    for c in df.columns
])

# Add a monotonically increasing ID column
df_dim = df_upper.withColumn("LHID", monotonically_increasing_id())

# Write to the gold layer dimension table (replaces it if it already exists)
df_dim.write.mode("overwrite").saveAsTable("gold.dim_branches")

# Verify
# display(df_dim)


In [0]:
# Read the silver table, uppercase string column values, alias every column name to UPPERCASE,
# replace the foreign keys (ACCOUNT_ID, PRODUCT_ID, BRANCH_ID) with the dimension LHID keys,
# and add a monotonically increasing ID column
from pyspark.sql.functions import upper, col, monotonically_increasing_id

df = spark.table("silver.transactions")

df_upper = df.select([
    upper(c).alias(c.upper()) if df.schema[c].dataType.simpleString() == "string" else col(c).alias(c.upper())
    for c in df.columns
])

# Load the dimension tables, keeping only their join key and LHID
dim_customers = spark.table("gold.dim_customers").select(
    col("ACCOUNT_ID").alias("_CUST_KEY"), col("LHID").alias("CUSTOMER_LHID"))
dim_products = spark.table("gold.dim_products").select(
    col("PRODUCT_ID").alias("_PROD_KEY"), col("LHID").alias("PRODUCT_LHID"))
dim_branches = spark.table("gold.dim_branches").select(
    col("BRANCH_ID").alias("_BRANCH_KEY"), col("LHID").alias("BRANCH_LHID"))

# Join transactions to each dimension on its key and replace the raw id columns
# with the corresponding dimension LHID columns
df_fact = (
    df_upper
    .join(dim_customers, col("ACCOUNT_ID") == col("_CUST_KEY"), "left")
    .join(dim_products, col("PRODUCT_ID") == col("_PROD_KEY"), "left")
    .join(dim_branches, col("BRANCH_ID") == col("_BRANCH_KEY"), "left")
    .drop("ACCOUNT_ID", "PRODUCT_ID", "BRANCH_ID",
          "_CUST_KEY", "_PROD_KEY", "_BRANCH_KEY")
    .withColumn("LHID", monotonically_increasing_id())
)

# Write to the gold layer fact table (replaces it if it already exists)
df_fact.write.mode("overwrite").saveAsTable("gold.fact_transactions")

# Verify
# display(df_fact)